In [ ]:
# --- Cell: imports ---
import os
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import sys
sys.path.append('.')  # folder containing vti_magnetic_import.py
from vti_magnetic_import import load_vti_sample, inspect_vti

from scipy.ndimage import gaussian_filter, map_coordinates
from skimage.measure import marching_cubes
import pyvista as pv

import matplotlib.ticker as mticker
import pandas as pd

import json

import Functions_3D as EB

from scipy.optimize import curve_fit

from skimage.measure import find_contours

def _gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu) ** 2 / (2 * sigma ** 2))



from scipy import ndimage as ndi

In [ ]:
def load_sample_data(save_dir, sample_name, date, date_v):
    """
    Load back the volume, wall_df, and metadata for one sample, as saved
    by save_sample_data.
    """
    volume_path = os.path.join(save_dir, f'{sample_name}_volume_{date_v}.npz')
    grad_path = os.path.join(save_dir, f'{sample_name}_grad_{date_v}.npz')
    curl_path = os.path.join(save_dir, f'{sample_name}_curl_{date_v}.npz')
    energies_path = os.path.join(save_dir, f'{sample_name}_energies_{date_v}.npz')
    df_path = os.path.join(save_dir, f'{sample_name}_wall_df_{date}.csv')
    zdf_path = os.path.join(save_dir, f'{sample_name}_z0_df_{date}.csv')
    meta_path = os.path.join(save_dir, f'{sample_name}_volume_analysis_metadata_{date_v}.json')

    npz = np.load(volume_path)
    cropped = {
        'Mx': npz['Mx'], 'My': npz['My'], 'Mz': npz['Mz'],
        'magnitude': npz['magnitude'],
        'shape': tuple(npz['shape']),
    }
    spacing = tuple(npz['spacing'])
    z_offset = 0 #int(npz['z_offset'])

    wall_df = pd.read_csv(df_path)

    z0_df = pd.read_csv(zdf_path)

    npz_grad = np.load(grad_path)
    grad = {'grad_xy': npz_grad['grad_xy'], 'grad_3D': npz_grad['grad_3D']}

    npz_curl = np.load(curl_path)
    curl = {'curl_x': npz_curl['curl_x'], 'curl_y': npz_curl['curl_y'], 'curl_z': npz_curl['curl_z'], 'curl_magnitude': npz_curl['curl_magnitude'], 'helicity': npz_curl['helicity'], 'parallel': npz_curl['parallel'], 'perpendicular': npz_curl['perpendicular'], 'par_perp': npz_curl['par_perp']}

    npz_energies = np.load(energies_path)
    energies = {'dmi': npz_energies['iDMI']}

    with open(meta_path) as f:
        metadata = json.load(f)

    return cropped, grad, curl, energies, wall_df, z0_df, spacing,z_offset, metadata

In [ ]:
# sample_names = ['270', '105', '665', '452'] 
# # dates_volume = ['14AUG26_interpolated', '22AUG26_interpolated', '22AUG26_interpolated', '14AUG26_interpolated']
# # dates = ['Interpolated_6OCT26', 'Interpolated_6OCT26', 'Interpolated_6OCT26', 'Interpolated_6OCT26']
# dates_volume = ['full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26']
# dates = ['full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26', 'full_volume_mag_const_sqrt(2)_nointerp_OCT26']
# voxel_nm = 8/np.sqrt(2)
# sample_colors = ['deeppink', 'cyan', 'gold', 'limegreen']
# sample_names_plot = ['[Co(0.2 nm)/ Ni(0.7 nm)]', '[Co(1.0 nm)/ Pt(0.5 nm)]', '[Co(0.6 nm)/ Ni(0.6 nm)/ Pt(0.5 nm)]', '[Co(0.4 nm)/ Ni(0.5 nm)/ Pt(0.2 nm)]'] 

# sample_names = ['105', '665', '452'] 
# dates_volume = ['mag_const_TV_interpolated_t_OCT26', 'mag_const_TV_interpolated_t_OCT26', 'mag_const_TV_interpolated_OCT26']
# dates = ['mag_const_TV_interpolated_t_OCT26', 'mag_const_TV_interpolated_t_OCT26', 'mag_const_TV_interpolated_OCT26']
# sample_colors = ['cyan', 'gold', 'limegreen']
# sample_names_plot = ['[Co(0.2 nm)/ Ni(0.7 nm)]', '[Co(1.0 nm)/ Pt(0.5 nm)]', '[Co(0.6 nm)/ Ni(0.6 nm)/ Pt(0.5 nm)]', '[Co(0.4 nm)/ Ni(0.5 nm)/ Pt(0.2 nm)]'] 


# sample_names = ['452', '452', '452'] 
# dates_volume = ['exactsubset_mag_const_TV_interpolated_OCT26', 'exactsubset_mag_const_interpolated_OCT26', 'mag_const_updated_interpolation']
# dates = ['exactsubset_mag_const_TV_interpolated_OCT26', 'exactsubset_mag_const_interpolated_OCT26', 'mag_const_updated_interpolation'] #, 'Interpolated_6OCT26', 'Interpolated_6OCT26']
# sample_colors = ['cyan', 'gold', 'limegreen']
# sample_names_plot = ['452 mag + TV', '452 mag new', '452 mag old'] 

# sample_names = ['452', '452'] 
# dates_volume = ['cropped_mag_const_2interp_bsd_OCT26', 'mag_const_updated_interpolation']
# dates = ['cropped_mag_const_2interp_bsd_OCT26', 'mag_const_updated_interpolation'] #, 'Interpolated_6OCT26', 'Interpolated_6OCT26']
# sample_colors = ['cyan', 'gold']
# sample_names_plot = ['452 mag new', '452 mag old'] 
# voxel_nm = 2

# sample_names = ['452', '452'] 
# dates_volume = ['exactsubset_mag_const_TV_interpolated_OCT26', 'mag_const_TV_interpolated_OCT26']
# dates = ['exactsubset_mag_const_TV_interpolated_OCT26', 'mag_const_TV_interpolated_OCT26'] #, 'Interpolated_6OCT26', 'Interpolated_6OCT26']
# sample_colors = ['cyan', 'darkcyan']
# sample_names_plot = ['452 mag + TV small', '452 mag + TV large'] 


sample_names = ['270', '105', '665', '452'] 
dates_volume = ['TV_8nm_NOV26', 'TV_full_8nm_NOV26', 'TV_8nm_NOV26', 'TV_8nm_NOV26', 'TV_8nm_NOV26'] 
dates = ['TV_8nm_NOV26', 'TV_8nm_NOV26', 'TV_8nm_NOV26', 'TV_8nm_NOV26', 'TV_8nm_NOV26'] 
sample_colors = ['deeppink', 'cyan', 'gold', 'limegreen']
sample_names_plot = ['[Co(0.2 nm)/ Ni(0.7 nm)]', '[Co(1.0 nm)/ Pt(0.5 nm)]', '[Co(0.6 nm)/ Ni(0.6 nm)/ Pt(0.5 nm)]', '[Co(0.4 nm)/ Ni(0.5 nm)/ Pt(0.2 nm)]'] 
sample_thickness = [66, 45, 51, 33]
voxel_nm = 8.0


all_wall_dfs = []
all_z0_df = []
volumes = {}
grad = {}
curl_dict = {}
neel_dict = {}

ct = 0
for name, date, date_v in zip(sample_names, dates, dates_volume):
    print(name)
    data_dir = f'C:/Users/emmab/Projects/Vector_Tomography/data_dir/{name}/'
    cropped, gradient, curl, energies, wall_df, z0_df, spacing, z_offset, meta = load_sample_data(data_dir, name, date, date_v)
    volumes[name] = cropped
    grad[name] = gradient
    curl_dict[name] = curl
    neel_dict[name] = energies
    wall_df = wall_df.copy()
    wall_df['sample'] = sample_names_plot[ct]  # name        
    all_wall_dfs.append(wall_df)
    z0_df = z0_df.copy()
    z0_df['sample'] = sample_names_plot[ct]  # name          
    all_z0_df.append(z0_df)
    ct = ct + 1

combined_df = pd.concat(all_wall_dfs, ignore_index=True)
combined_z0_df = pd.concat(all_z0_df, ignore_index=True)
# Now you can group by sample_name for direct comparison, e.g.:
#combined_df.groupby('sample_name')['psi_chiral'].agg(['mean', 'std', 'count'])


figure_name = {
    'psi_unsigned': 'ψ',
    'psi_unsigned': 'ψ', 
    'M_normal': '||Surface Normal||', 
    'Exchange': '(∇·M)$^2$',
    'curl_magnitude': '||∇×M||',
    'PMA': 'PMA',
    'iDMI': 'iDMI',
}

keff_df = pd.read_csv(r'C:/Users/emmab/Projects/Vector_Tomography/Magnetometry_params_NOV26.csv')

# Link tomography samples to Keff samples via thickness
meta_df = pd.DataFrame({
    'sample_name':  sample_names,
    'sample':       sample_names_plot,       # matches combined_df['sample']
    'thickness_nm': sample_thickness,
})
meta_df = meta_df.merge(keff_df, on='thickness_nm', how='left', validate='one_to_one')

# Check that every sample matched
missing = meta_df[meta_df['Keff'].isna()]
if len(missing):
    print("No Keff match for:", missing['sample_name'].tolist())

# Lookup by tomography sample name, e.g. keff['270']['Q']
keff = meta_df.set_index('sample_name').to_dict('index')

# Attach to the combined dataframes (columns: Keff, Q, Ms, Hc, ...)
keff_cols = ['sample', 'thickness_nm', 'Ms', 'Hc', 'Mr', 'Keff', 'Q', 'D_thr']
keff_cols = [c for c in keff_cols if c in meta_df.columns]
combined_df    = combined_df.merge(meta_df[keff_cols], on='sample', how='left')
combined_z0_df = combined_z0_df.merge(meta_df[keff_cols], on='sample', how='left')



In [ ]:
grad['270'].keys()

# Curl 

In [ ]:
# import json
# import numpy as np
# import matplotlib.pyplot as plt

# # ---------------- helpers ----------------
# def _mean_median_std(z, w, voxel_nm):
#     """Weighted mean, median (interpolated within the crossing layer), and std of z."""
#     w = np.where(np.isfinite(w), w, 0.0)
#     tot = w.sum()
#     if tot <= 0:
#         return np.nan, np.nan, np.nan
#     p = w / tot
#     mean = float(np.sum(z * p))
#     cum = np.cumsum(p)
#     k = min(int(np.searchsorted(cum, 0.5)), len(z) - 1)
#     if p[k] == 0:
#         median = float(z[k])
#     else:
#         median = float(z[k] - voxel_nm / 2 + (0.5 - (cum[k] - p[k])) / p[k] * voxel_nm)
#     std = float(np.sqrt(np.sum(p * (z - mean) ** 2)))
#     return mean, median, std

# def _block_sums(a, valid, block):
#     """Per-block, per-layer sums of a and of the voxel count. Arrays are (z, x, y).
#     Returns two (n_blocks, nz) arrays."""
#     nz, nx, ny = a.shape
#     px, py = (-nx) % block, (-ny) % block
#     pad = ((0, 0), (0, px), (0, py))
#     def tile(b):
#         b = np.pad(b, pad)
#         b = b.reshape(nz, (nx + px) // block, block, (ny + py) // block, block).sum(axis=(2, 4))
#         return b.reshape(nz, -1).T
#     return tile(a), tile(valid.astype(float))

# # ---------------- main function ----------------
# def plot_curl_vs_z(curls, sample_names_plot, sample_colors,
#                    curl_key='curl_magnitude', title=None, voxel_nm=8.0,
#                    use_abs=True, reduce='sum', normalize_bins=True,
#                    block_px=8, n_boot=500, plot_individual=True, combined_plot=True,
#                    save_path=None, figure_name=None, seed=0):
#     """
#     curls       : {sample_name: curl_dict}, each array (z, x, y)
#     wall_masks  : {sample_name: bool array (z, x, y)} or None (use every voxel)
#     curl_key    : 'curl_magnitude', 'parallel', 'perpendicular', 'helicity', ...
#     use_abs     : take |value| first (needed for signed keys, since weights must be >= 0)
#     reduce      : 'sum' (mixes strength with wall area) or 'mean' (per masked voxel)
#     normalize_bins : bars show the fraction of total weight per layer instead of raw values
#     block_px    : bootstrap tile size in array pixels (8 px = 64 nm at 8 nm pixels;
#                   use ~32 for a 2 nm grid). Should exceed the correlation length
#                   (about a wall width).
#     n_boot      : bootstrap replicates (lower it if memory is tight on large volumes)
#     save_path   : optional .json file for the stats of all samples
#     """
#     rng = np.random.default_rng(seed)
#     results, plot_data = {}, []

#     ylab = {'sum': f'Sum of {curl_key} per layer', 'mean': f'Mean {curl_key} per voxel'}[reduce]
#     ylab_bar = (f'Fraction of total {curl_key}' if normalize_bins else ylab)

#     for name, color in zip(sample_names_plot, sample_colors):
#         a = np.asarray(curls[name][curl_key], dtype=float)
#         if use_abs:
#             a = np.abs(a)
#         m = np.ones(a.shape, bool) 
#         valid = m & np.isfinite(a)
#         a = np.where(valid, a, 0.0)

#         nz = a.shape[0]
#         z = (np.arange(nz) - (nz - 1) / 2) * voxel_nm          # layer centers rel. to film center
#         num = a.sum(axis=(1, 2))
#         den = valid.sum(axis=(1, 2)).astype(float)
#         vals = num if reduce == 'sum' else np.where(den > 0, num / np.maximum(den, 1), 0.0)
#         frac = vals / vals.sum() if vals.sum() > 0 else np.full(nz, np.nan)

#         mean, median, std = _mean_median_std(z, vals, voxel_nm)
#         p = vals / vals.sum()
#         skew = float(np.sum(p * ((z - mean) / std) ** 3)) if std > 0 else np.nan

#         # ---- block bootstrap over (x, y) tiles ----
#         bn, bd = _block_sums(a, valid, block_px)
#         keep = bd.sum(axis=1) > 0                              # drop tiles with no wall voxels
#         bn, bd = bn[keep], bd[keep]
#         nb = len(bn)
#         layer_err = np.full(nz, np.nan)
#         st = dict(sterr_mean=np.nan, sterr_median=np.nan, sterr_std=np.nan,
#                   ci95_mean=(np.nan, np.nan), ci95_median=(np.nan, np.nan))
#         if nb >= 5:
#             cnt = rng.multinomial(nb, np.full(nb, 1.0 / nb), size=n_boot).astype(float)
#             nb_, db_ = cnt @ bn, cnt @ bd
#             vb = nb_ if reduce == 'sum' else np.where(db_ > 0, nb_ / np.maximum(db_, 1), 0.0)
#             boot = np.array([_mean_median_std(z, v, voxel_nm) for v in vb])   # (n_boot, 3)
#             layer_err = vb.std(axis=0)
#             st = dict(sterr_mean=float(np.nanstd(boot[:, 0])),
#                       sterr_median=float(np.nanstd(boot[:, 1])),
#                       sterr_std=float(np.nanstd(boot[:, 2])),
#                       ci95_mean=tuple(np.nanpercentile(boot[:, 0], [2.5, 97.5])),
#                       ci95_median=tuple(np.nanpercentile(boot[:, 1], [2.5, 97.5])))

#         stats = dict(curl_key=curl_key, reduce=reduce, use_abs=use_abs, voxel_nm=voxel_nm,
#                      block_px=block_px, n_boot=n_boot, n_blocks=int(nb),
#                      n_voxels=int(valid.sum()), total=float(vals.sum()),
#                      mean=mean, median=median, std=std, variance=std ** 2, stderr = std/ np.sqrt(num),  skew=skew,
#                      layer_fractions=frac.tolist(), **st)
#         results[name] = dict(z_nm=z, layer_values=vals, layer_err=layer_err,
#                              fractions=frac, stats=stats)
#         plot_data.append(dict(name=name, color=color, z=z, vals=vals, err=layer_err,
#                               frac=frac, stats=stats))

#         print(f"{name}: mean = {mean:.2f} ± {st['sterr_mean']:.2f} nm, "
#               f"median = {median:.2f} ± {st['sterr_median']:.2f} nm, "
#               f"std = {std:.2f} nm, skew = {skew:.2f}, fractions = {np.round(frac, 3)}")

#     # ---------------- plotting ----------------
#     def draw(ax1, ax2, d, line_alpha, bar_alpha, stat_colors=None):
#         ax1.errorbar(d['z'], d['vals'], yerr=d['err'], fmt='o-', capsize=3,
#                      color=d['color'], alpha=line_alpha, label=d['name'])
#         ax2.bar(d['z'], d['frac'] if normalize_bins else d['vals'], width=0.9 * voxel_nm,
#                 color=d['color'], alpha=bar_alpha, edgecolor='gray', label=d['name'])
#         s = d['stats']
#         cm, cmed = stat_colors if stat_colors else (d['color'], d['color'])
#         for ax in (ax1, ax2):
#             ax.axvline(s['mean'], color=cm, lw=1.5,
#                        label=f"mean = {s['mean']:.2f} ± {s['sterr_mean']:.2f} nm")
#             ax.axvline(s['median'], color=cmed, lw=1.5, ls='-.',
#                        label=f"median = {s['median']:.2f} ± {s['sterr_median']:.2f} nm")
#         ax2.plot([], [], ' ', label=f"std = {s['std']:.2f} nm")

#     def finish(fig, ax1, ax2, z, suptitle, fname=None):
#         for ax in (ax1, ax2):
#             ax.axvline(0, color='k', ls='--', lw=1, label='Film center')
#             ax.set_xticks(z)
#             ax.set_xticklabels([f'{v:g}' for v in z])
#             ax.set_xlabel('z relative to film center (nm)', fontsize=18)
#             ax.legend(loc='upper left', fontsize=12)
#         ax1.set_ylabel(ylab, fontsize=18)
#         ax2.set_ylabel(ylab_bar, fontsize=18)
#         fig.suptitle(suptitle, fontsize=20)
#         fig.tight_layout()
#         if fname:
#             fig.savefig(fname, dpi=200, bbox_inches='tight')
#         plt.show()

#     if plot_individual:
#         for d in plot_data:
#             fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 7), sharex=True)
#             draw(ax1, ax2, d, 1.0, 0.5, stat_colors=('red', 'tab:blue'))
#             fname = f"{figure_name}_{d['name']}.png" if figure_name else None
#             finish(fig, ax1, ax2, d['z'], f"{curl_key} vs. thickness position, {d['name']}\n{title or ''}", fname)

#     if combined_plot and plot_data:
#         fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(25, 15), sharex=True)
#         for d in plot_data:
#             draw(ax1, ax2, d, 0.8, 0.4)
#         z_all = plot_data[0]['z']
#         finish(fig, ax1, ax2, z_all, f"{curl_key} vs. thickness position, all samples\n{title or ''}",
#                f"{figure_name}_all.png" if figure_name else None)

#     if save_path:
#         with open(save_path, 'w') as f:
#             json.dump({k: v['stats'] for k, v in results.items()}, f, indent=2,
#                       default=lambda o: o.tolist() if isinstance(o, np.ndarray) else float(o))
#     return results

In [ ]:
# curl_res = plot_curl_vs_z(
#     curls={name: curl_dict[name] for name in sample_names},
#     sample_names_plot=sample_names, sample_colors=sample_colors,
#     curl_key='parallel', title='Bloch (curl·m) per layer',
#     voxel_nm=8.0, reduce='mean', block_px=8)

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

def _z_mean_median(z, w, voxel_nm):
    """Mean and median of z, using the per-layer mean curl as weights.
    Median is interpolated within the layer where the cumulative weight crosses 0.5."""
    w = np.nan_to_num(w)
    if w.sum() <= 0:
        return np.nan, np.nan
    p = w / w.sum()
    mean = float(np.sum(z * p))
    cum = np.cumsum(p)
    k = min(int(np.searchsorted(cum, 0.5)), len(z) - 1)
    median = float(z[k] - voxel_nm / 2 + (0.5 - (cum[k] - p[k])) / p[k] * voxel_nm) if p[k] > 0 else float(z[k])
    return mean, median

def curl_layer_stats(curls,names, colors, curl_key='parallel',
                     voxel_nm=8.0, use_abs=False, plot_individual=False,
                     plot_combined=True, save_path=None):
    """
    curls      : {name: curl_dict}, arrays (z, x, y)
    wall_masks : {name: bool array (z, x, y)} or None (all voxels)
    curl_key   : 'curl_magnitude', 'parallel', 'perpendicular', ...
    use_abs    : average |value| (signed keys cancel otherwise, and the
                 z mean/median need non-negative weights)
    """
    results = {}
    for name in names:
        a = np.asarray(curls[name][curl_key], float)
        if use_abs:
            a = np.abs(a)
        m = np.ones(a.shape, bool) 
        m = m & np.isfinite(a)

        nz = a.shape[0]
        z = (np.arange(nz) - (nz - 1) / 2) * voxel_nm      # layer centers rel. to film center
        n, mean, std, = np.zeros(nz, int), np.full(nz, np.nan), np.full(nz, np.nan)
        # mean_at_center, std_at_center= np.zeros(nz, int), np.zeros(nz, int)
        k_cent = nz//2
        for k in range(nz):
            v = a[k][m[k]]
            n[k] = v.size
            if v.size > 1:
                mean[k], std[k] = v.mean(), v.std(ddof=1)
                if k == k_cent: 
                    print(name)
                    print(k) 
                    print(v.mean())
                    mean_at_center, std_at_center = v.mean(), v.std(ddof=1)
        sterr = std / np.sqrt(np.maximum(n, 1))

        z_mean, z_median = _z_mean_median(z, mean, voxel_nm)
        v_all = a[m]
        scalars = {
            f'{curl_key}_mean':     float(v_all.mean()),
            f'{curl_key}_std':      float(v_all.std(ddof=1)),
            f'{curl_key}_sterr':    float(v_all.std(ddof=1) / np.sqrt(v_all.size)),
            f'{curl_key}_z_mean':   z_mean,      # nm from film center
            f'{curl_key}_z_median': z_median,    # nm from film center
            f'{curl_key}_mean_at_center': mean_at_center,    # nm from film center
            f'{curl_key}_std_at_center': std_at_center, 
        }
        results[name] = dict(curl_key=curl_key, use_abs=use_abs, voxel_nm=voxel_nm,
                             z_nm=z, n=n, mean=mean, std=std, sterr=sterr, scalars=scalars)

        # print(f"{name}: overall mean = {scalars[f'{curl_key}_mean']:.4g}, "
        #       f"z mean = {z_mean:.2f} nm, z median = {z_median:.2f} nm")
        # print("  layer means:", np.round(mean, 4))

    def draw(ax, name, color, alpha=1.0):
        r = results[name]
        s = r['scalars']
        ax.errorbar(r['z_nm'], r['mean'], yerr=r['sterr'], fmt='o-', capsize=5,
                    elinewidth=1.5, color=color, alpha=alpha, label=name)
        ax.axvline(s[f'{curl_key}_z_mean'], color=color, lw=1.5, ls='-', alpha=alpha,
                   label=f"mean = {s[f'{curl_key}_z_mean']:.2f} nm")
        ax.axvline(s[f'{curl_key}_z_median'], color=color, lw=1.5, ls='-.', alpha=alpha,
                   label=f"median = {s[f'{curl_key}_z_median']:.2f} nm")

    def finish(ax, title):
        ax.axvline(0, color='k', ls='--', lw=1, label='Film center')
        ax.set_xlabel('z relative to film center (nm)', fontsize=16)
        bars = '|' if use_abs else ''
        ax.set_ylabel(f"mean {bars}{curl_key}{bars}", fontsize=16)
        # ax.set_title(title, fontsize=14)
        ax.legend(fontsize=11)
        plt.tight_layout(); plt.show()

    if plot_individual:
        for name, c in zip(names, colors):
            fig, ax = plt.subplots(figsize=(7, 5))
            draw(ax, name, c)
            finish(ax, f"{curl_key} vs. z, {name}\n(error bars = ± stderr)")

    if plot_combined:
        fig, ax = plt.subplots(figsize=(9, 6))
        for name, c in zip(names, colors):
            draw(ax, name, c, alpha=0.85)
        finish(ax, f"{curl_key} vs. z, all samples\n(error bars = ± stderr)")

    if save_path:
        with open(save_path, 'w') as f:
            json.dump(results, f, indent=2,
                      default=lambda o: o.tolist() if isinstance(o, np.ndarray) else float(o))
    return results

In [ ]:
par_res = curl_layer_stats(curl_dict, sample_names, sample_colors,
                           curl_key='parallel', voxel_nm=8.0)
mag_res = curl_layer_stats(curl_dict, sample_names, sample_colors,
                           curl_key='curl_magnitude', voxel_nm=8.0)

In [ ]:
hel_res = curl_layer_stats(curl_dict, sample_names, sample_colors,
                           curl_key='helicity', voxel_nm=8.0)

In [ ]:
hel_res['270']['mean']

In [ ]:
hel_res['270']['scalars']['helicity_mean_at_center']

# defect analysis

In [ ]:
S8 = np.ones((3, 3), bool)   # 8-connectivity

def helicity_switches_2x(curl_dict, names, labels=None,
                      curl_key='helicity', voxel_nm=8.0, z_index=None,
                      deadband_frac=1, min_size=1, plot=True):
    """
    curl_dict     : {name: curl dict}, arrays (z, x, y)
    names         : keys of curl_dict to process
    labels        : optional display names, same order as `names`
    curl_key      : signed key, 'helicity' (curl·m) or 'parallel'
    z_index       : slice to use; default is the central layer, nz // 2
    deadband_frac : |h| below deadband_frac * median(|h|) is treated as 0
    min_size      : clusters smaller than this many pixels are dropped (1 = keep all)
    Switches are estimated as 2 x (number of clusters of each sign).
    """
    results = {}
    for i, name in enumerate(names):
        label = labels[i] if labels is not None else name
        h = np.asarray(curl_dict[name][curl_key], float)
        nz, nx, ny = h.shape
        k = nz // 2 if z_index is None else z_index
        hk = np.nan_to_num(h[k])

        # ---- ternary map: +1 / 0 / -1 ----
        thr = deadband_frac * np.median(np.abs(hk))
        pos = hk > thr
        neg = hk < -thr
        tern = pos.astype(int) - neg.astype(int)

        # ---- cluster each sign separately ----
        labs, counts = {}, {}
        for sgn, mask in (('red', pos), ('blue', neg)):
            lab, n = ndi.label(mask, structure=S8)
            if n and min_size > 1:
                sizes = np.bincount(lab.ravel())[1:]
                keep = np.flatnonzero(sizes >= min_size) + 1
                lab, n = ndi.label(np.isin(lab, keep), structure=S8)
            labs[sgn], counts[sgn] = lab, int(n)

        n_red, n_blue = counts['red'], counts['blue']
        pix_um = voxel_nm * 1e-3
        fov_um2 = nx * ny * pix_um**2

        scalars = {
            'hel_switch_n_red':          n_red,
            'hel_switch_n_blue':         n_blue,
            'hel_switch_red_x2':         2 * n_red,
            'hel_switch_blue_x2':        2 * n_blue,
            'hel_switch_red_x2_per_um2':  2 * n_red / fov_um2,
            'hel_switch_blue_x2_per_um2': 2 * n_blue / fov_um2,
            'hel_switch_fov_um2':        float(fov_um2),
            'hel_switch_deadband':       float(deadband_frac),
            'hel_switch_min_size':       int(min_size),
            'hel_switch_z_index':        int(k),
        }
        results[name] = dict(label=label, scalars=scalars, tern=tern, cluster_labels=labs)
        print(f"{label}: red = {n_red} (x2 = {2*n_red}, {2*n_red/fov_um2:.2f} per µm²), "
              f"blue = {n_blue} (x2 = {2*n_blue}, {2*n_blue/fov_um2:.2f} per µm²), "
              f"slice {k}, FOV = {fov_um2:.2f} µm²")

        # ---- plot with numbered clusters ----
        if plot:
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.set_facecolor('0.9')
            ext = (0, nx * pix_um, 0, ny * pix_um)
            ax.imshow(np.ma.masked_where(tern == 0, tern).T, origin='lower', extent=ext,
                      cmap='coolwarm', vmin=-1, vmax=1, interpolation='nearest')
            for sgn, prefix, tc in (('red', 'R', 'darkred'), ('blue', 'B', 'navy')):
                lab, n = labs[sgn], counts[sgn]
                if n:
                    cms = ndi.center_of_mass(lab > 0, lab, range(1, n + 1))
                    for j, (cx, cy) in enumerate(cms, start=1):
                        ax.text((cx + 0.5) * pix_um, (cy + 0.5) * pix_um, f"{prefix}{j}",
                                fontsize=8, ha='center', va='center', color=tc,
                                bbox=dict(boxstyle='round,pad=0.1', fc='white',
                                          ec='none', alpha=0.7))
            ax.set_xlabel('x (µm)', fontsize=14)
            ax.set_ylabel('y (µm)', fontsize=14)
            ax.set_title(f"{label}\nred = {n_red} (x2 = {2*n_red}), blue = {n_blue} "
                         f"(x2 = {2*n_blue})\nslice {k}, deadband {deadband_frac:g}", fontsize=13)
            plt.tight_layout()
            plt.show()

    return results


def helicity_switches(curl_dict, names, labels=None,
                      curl_key='helicity', voxel_nm=8.0, z_index=None,
                      deadband_frac=1, plot=True):
    """
    curl_dict     : {name: curl dict}, arrays (z, x, y)
    names         : keys of curl_dict to process
    labels        : optional display names, same order as `names` (for plot titles)
    curl_key      : signed key, 'helicity' (curl·m) or 'parallel'
    z_index       : slice to use; default is the central layer, nz // 2
    deadband_frac : |h| below deadband_frac * median(|h| in mask) is treated as 0
    """
    results = {}
    for i, name in enumerate(names):
        label = labels[i] if labels is not None else name
        h = np.asarray(curl_dict[name][curl_key], float)
        nz, nx, ny = h.shape
        k = nz // 2 if z_index is None else z_index

        hk = np.nan_to_num(h[k])
        mk = np.ones(hk.shape, bool) 

        # ---- ternary map: +1 / 0 / -1 ----
        thr = deadband_frac * np.median(np.abs(hk[mk]))
        pos = mk & (hk > thr)
        neg = mk & (hk < -thr)
        tern = pos.astype(int) - neg.astype(int)

        # ---- pixels where + and - touch (8-connectivity), clustered into events ----
        # touch = (pos & ndi.binary_dilation(neg, S8)) | (neg & ndi.binary_dilation(pos, S8))
        # lab, n = ndi.label(touch, structure=S8)

        r = 1
        pos_d = ndi.binary_dilation(pos, S8, iterations=r) & mk
        neg_d = ndi.binary_dilation(neg, S8, iterations=r) & mk
        touch = pos_d & neg_d                      # overlap region = one blob per switch
        lab, n = ndi.label(touch, structure=S8)



        # ---- normalize by FOV ----
        pix_um = voxel_nm * 1e-3
        fov_um2 = nx * ny * pix_um**2
        scalars = {
            'hel_switch_count':        int(n),
            'hel_switch_per_um2':      float(n / fov_um2),
            'hel_switch_fov_um2':      float(fov_um2),
            'hel_switch_deadband':     float(deadband_frac),
            'hel_switch_z_index':      int(k),
        }
        results[name] = dict(label=label, scalars=scalars, tern=tern, interface_labels=lab)
        print(f"{label}: {n} switches in slice {k}, FOV = {fov_um2:.2f} µm², "
              f"{n / fov_um2:.2f} per µm²")

        # ---- plot with numbered interfaces ----
        if plot:
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.set_facecolor('0.9')
            ext = (0, nx * pix_um, 0, ny * pix_um)
            img = np.ma.masked_where(tern == 0, tern)
            ax.imshow(img.T, origin='lower', extent=ext, cmap='coolwarm',
                      vmin=-1, vmax=1, interpolation='nearest')
            if n > 0:
                cms = ndi.center_of_mass(touch, lab, range(1, n + 1))
                for j, (cx, cy) in enumerate(cms, start=1):
                    ax.plot((cx + 0.5) * pix_um, (cy + 0.5) * pix_um, 'k.', ms=3)
                    ax.text((cx + 0.5) * pix_um, (cy + 0.5) * pix_um, str(j),
                            fontsize=9, ha='center', va='bottom', color='k',
                            bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='none', alpha=0.7))
            ax.set_xlabel('x (µm)', fontsize=14)
            ax.set_ylabel('y (µm)', fontsize=14)
            ax.set_title(f"{label}\n{n} switches, {n / fov_um2:.2f} per µm²  "
                         f"(slice {k}, deadband {deadband_frac:g})", fontsize=13)
            plt.tight_layout()
            plt.show()

    return results

In [ ]:
sw2x_res = helicity_switches_2x(curl_dict, sample_names,
                           labels=sample_names_plot, curl_key='helicity',
                           voxel_nm=8.0, deadband_frac=3)

In [ ]:
sw_res = helicity_switches(curl_dict, sample_names,
                           labels=sample_names_plot, curl_key='helicity',
                           voxel_nm=8.0, deadband_frac=3)

# Neel wall (top and bottom DMI count per fov)

In [ ]:
def _layer_and_mask(entry, key, spec):
    """Return (2D signed map, 2D bool mask) for a layer spec (int index or dict key)."""
                                      # index into a 3D array
    a = np.asarray(entry[key], float)[spec]
    m = np.ones(a.shape, bool) 
    return np.nan_to_num(a), m

def neel_chirality_clusters(neel_dict, names,labels=None,
                            key='dmi', layers=None, voxel_nm=8.0,
                            deadband_frac=0.2, min_size=4,
                            plot=True, number_clusters=True):
    """
    neel_dict     : {name: dict}; entry[key] is a signed Néel-chirality array (z, x, y)
    layers        : {'bottom': 0, 'top': -1} by default. Values can be ints (index into
                    entry[key]) or strings (keys of separate 2D arrays in entry)
    deadband_frac : |value| below deadband_frac * median(|value| in mask) counts as 0
    min_size      : clusters smaller than this many pixels are dropped (removes specks)
    Red = positive, blue = negative (coolwarm).
    """
    layers = layers or {'bottom': 0, 'top': -1}
    results = {}

    for i, name in enumerate(names):
        label = labels[i] if labels is not None else name
        scalars, maps = {}, {}

        for lname, spec in layers.items():
            a, m = _layer_and_mask(neel_dict[name], key, spec)
            nx, ny = a.shape
            thr = deadband_frac * np.median(np.abs(a[m])) #if m.any() else 0.0
            pos, neg = m & (a > thr), m & (a < -thr)

            counts, areas, labs = {}, {}, {}
            for sgn, mask in (('red', pos), ('blue', neg)):
                lab, n = ndi.label(mask, structure=S8)
                if n and min_size > 1:
                    sizes = np.bincount(lab.ravel())[1:]
                    keep = np.flatnonzero(sizes >= min_size) + 1
                    lab, n = ndi.label(np.isin(lab, keep), structure=S8)
                counts[sgn], areas[sgn], labs[sgn] = int(n), int((lab > 0).sum()), lab

            fov_um2 = nx * ny * (voxel_nm * 1e-3) ** 2
            tot = counts['red'] + counts['blue']
            p = f'neel_{lname}'
            scalars.update({
                f'{p}_n_red':          counts['red'],
                f'{p}_n_blue':         counts['blue'],
                f'{p}_n_total':        tot,
                f'{p}_red_fraction':   counts['red'] / tot if tot else np.nan,
                f'{p}_n_red_per_um2':  counts['red'] / fov_um2,
                f'{p}_n_blue_per_um2': counts['blue'] / fov_um2,
                f'{p}_area_red_frac':  areas['red'] / max(areas['red'] + areas['blue'], 1),
            })
            scalars.update({'neel_fov_um2': float(fov_um2),
                            'neel_deadband': float(deadband_frac),
                            'neel_min_size': int(min_size)})
            maps[lname] = dict(pos=pos, neg=neg, labs=labs, counts=counts, shape=(nx, ny))
            print(f"{label} [{lname}]: red = {counts['red']}, blue = {counts['blue']}, "
                  f"red fraction = {scalars[f'{p}_red_fraction']:.2f}")

        results[name] = dict(label=label, scalars=scalars, maps=maps)

        if plot:
            fig, axes = plt.subplots(1, len(layers), figsize=(7 * len(layers), 6), squeeze=False)
            for ax, (lname, idx) in zip(axes[0], layers.items()):
                a = np.nan_to_num(np.asarray(neel_dict[name][key][idx], float))
                nx, ny = a.shape
                pix = voxel_nm * 1e-3
                vmax = np.percentile(np.abs(a), 99)
                im = ax.imshow(a.T, origin='lower', extent=(0, nx * pix, 0, ny * pix),
                            cmap='turbo', vmin=-vmax, vmax=vmax, interpolation='nearest')
                fig.colorbar(im, ax=ax, shrink=0.8)
                ax.set_title(f"{label}, {lname}", fontsize=13)
                ax.set_xlabel('x (µm)'); ax.set_ylabel('y (µm)')
            plt.tight_layout(); plt.show()


            fig, axes = plt.subplots(1, len(layers), figsize=(8 * len(layers), 8), squeeze=False)
            for ax, (lname, d) in zip(axes[0], maps.items()):
                nx, ny = d['shape']
                pix = voxel_nm * 1e-3
                ext = (0, nx * pix, 0, ny * pix)
                tern = d['pos'].astype(int) - d['neg'].astype(int)
                ax.set_facecolor('0.9')
                ax.imshow(np.ma.masked_where(tern == 0, tern).T, origin='lower', extent=ext,
                          cmap='coolwarm', vmin=-1, vmax=1, interpolation='nearest')
                if number_clusters:
                    for sgn, tc in (('red', 'darkred'), ('blue', 'navy')):
                        lab, n = d['labs'][sgn], d['counts'][sgn]
                        if n:
                            for j, (cx, cy) in enumerate(ndi.center_of_mass(lab > 0, lab, range(1, n + 1)), 1):
                                ax.text((cx + .5) * pix, (cy + .5) * pix, str(j), fontsize=8, color=tc,
                                        ha='center', va='center',
                                        bbox=dict(boxstyle='round,pad=0.1', fc='white', ec='none', alpha=0.6))
                ax.set_title(f"{label}, {lname}\nred = {d['counts']['red']}, blue = {d['counts']['blue']}",
                             fontsize=13)
                ax.set_xlabel('x (µm)'); ax.set_ylabel('y (µm)')
            plt.tight_layout(); plt.show()

    return results

In [ ]:
neel_res = neel_chirality_clusters(neel_dict, sample_names,
                                   labels=sample_names_plot, key='dmi',
                                   layers={'bottom': 0, 'top': -1},
                                   deadband_frac=0.7, min_size=2)

# separate 2D arrays instead:
# layers={'bottom': 'neel_bottom', 'top': 'neel_top'}

# SURFACE NORMAL

In [ ]:
def film_geometry(wall_df, voxel_nm=2.0, ):
    """
   
    """
    z_min = wall_df['z'].min() 
    print(z_min)
    z_max = wall_df['z'].max() 

    z_center = 0.5 * (z_min + z_max)

    thickness_nm = (z_max - z_min) + voxel_nm ### not sure if this should be interpolated or voxel
    layers = np.arange(int(round(z_min / voxel_nm)), int(round(z_max / voxel_nm)) + 1)

    geom = dict(z_min=z_min, z_max=z_max, z_center=z_center,
                thickness=thickness_nm,
                z_bottom=z_center - thickness_nm / 2, z_top=z_center + thickness_nm / 2,
                voxel_nm=voxel_nm, layers=layers, x_vals=layers * voxel_nm - z_center)
    print(f"film: vertices span {z_min:.1f} to {z_max:.1f} nm, {len(layers)} layers, "
          f"thickness {thickness_nm:.1f} nm, center {z_center:.1f} nm")
    return geom

In [ ]:
def plot_Bloch_angle_vs_z(combined_df, combined_z0_df, sample_names, sample_names_plot, sample_colors, figure_name, bounds,
                          angle_col='M_n_n', bloch_axis=None, neel_axis=None,
                          title=None, max_gap=2.0, voxel_nm = 8.0, bin_nm = None,
                          plot_individual=True, show_fit=True, show_fit_summary=True, normalize_bins = True):

 
    def add_reference_lines(ax, stats=None):
        if bloch_axis is not None:
            for k, deg in enumerate(bloch_axis):
                ax.axhline(deg, color='magenta', label='Bloch Wall' if k == 0 else None)
        if neel_axis is not None:
            for k, deg in enumerate(neel_axis):
                ax.axhline(deg, color='cyan', label='Neel Wall' if k == 0 else None)
        if stats:
            if 'mu' in stats:
                ax.axvline(stats['mu'], color='darkorange', label=f"fit mu = {stats['mu']:.2f} nm")
            ax.axvline(stats['mean'], color='red', label=f"mean = {stats['mean']:.2f} nm")
            ax.axvline(stats['median'], color='tab:blue', label=f"median = {stats['median']:.2f} nm")

        
 
    def draw(ax, ax2, p, line_alpha, bar_alpha):
        g = p['grouped']
        ax.errorbar(p['x_line'], g['mean'], yerr=g['std'], fmt='o-', capsize=3,
                    color=p['color'], label=p['comp'], alpha=line_alpha)
        ax2.bar(p['x_bar'], p['counts'], width=p['width'], color=p['color'],
                alpha=bar_alpha, label=p['comp'])

    def draw_fit(ax2, p, zedges, stats = None):
        f = p['fit']
        if f is None:
            return
        xs = np.linspace(zedges[0], zedges[-1], 300)
        ax2.plot(xs, _gaussian(xs, f['A'], f['mu'], f['sigma']), color='k', lw=2)
        ax2.hlines(f['A'] / 2, f['mu'] - f['fwhm'] / 2, f['mu'] + f['fwhm'] / 2,
                color='k', lw=2.5, alpha=0)
        ax2.axvline(f['mu'], color='darkorange', lw=1.5, ls=':', label=f" mu = {f['mu']:.2f} nm \n FWHM = {f['fwhm']:.2f} nm")
        if stats:
            ax2.axvline(stats['mu'], color='darkorange', label= f'fit mu = {stats['mu']:.2f} nm')
            ax2.axvline(stats['mean'], color='red', label= f'mean = {stats['mean']:.2f} nm')
            ax2.axvline(stats['median'], color='tab:blue', label= f'median = {stats['median']:.2f} nm')

 
    def finish(ax, ax2, ticks, suptitle, plot_stats = None):
        #print(plot_stats)
        add_reference_lines(ax, stats = plot_stats)
        ax.axvline(0, color='k', linestyle='--', label='Wall center')
        ax.set_ylabel(angle_col, fontsize=15)
        ax.set_xlabel('z voxel layer (relative to center)', fontsize=15)
        ax.legend(loc = 'upper left', fontsize = 15)
 
        ax2.axvline(0, color='k', linestyle='--', label='Wall center')
        ax2.axvline(stats['mean'], color='red', label=f"mean = {stats['mean']:.2f} nm")
        ax2.axvline(stats['median'], color='tab:blue', label=f"median = {stats['median']:.2f} nm")
        ax2.set_xlabel('z voxel layer (relative to center)', fontsize=15)
        if normalize_bins: 
            ax2.set_ylabel(f'Fraction of {angle_col} = 0 crossings', fontsize=15)
        else:
            ax2.set_ylabel(f'# of {angle_col} = 0 crossings', fontsize=15)
        ax2.set_xticks(ticks)
        ax2.legend(loc = 'upper left', fontsize = 15)
 
        plt.gcf().suptitle(suptitle, fontsize=20)
        plt.tight_layout()
        plt.show()
 
    results = {}
    plot_data = []          # collected per-sample for the combined figure at the end
 
    for name, color, comp in zip(sample_names_plot, sample_colors, sample_names_plot):

        wall_df = combined_df[combined_df['sample'] == name].copy()
        d = wall_df[['z', angle_col]].copy()

        z0_df = combined_z0_df[combined_z0_df['sample'] == name].copy()

        if bin_nm is None:
            bin_nm = voxel_nm

        z_min, z_max = d['z'].min(), d['z'].max()
        z_center = 0.5 * (z_min + z_max)                    # film center (symmetric crop)
        T = (z_max - z_min) + voxel_nm                      # film thickness covered by the samples
        n_bins = int(round(T / bin_nm))

        geom = film_geometry(wall_df, voxel_nm)
        layers, x_vals, z_center = geom['layers'], geom['x_vals'], geom['z_center']
      
        # ---- fine z bins at the interpolated resolution, centered on the voxel centers ----
        d['z_layer'] = np.rint((d['z'] - z_min) / voxel_nm).astype(int)    # 0, 1, 2, ...
        layers = np.arange(d['z_layer'].max() + 1)
        x_vals = z_min + layers * voxel_nm - z_center                      # layer centers relative to film center
        grouped_f = d.groupby('z_layer')[angle_col].agg(['mean', 'std', 'count', 'sem']).reindex(layers)
        grouped_f['z_rel_nm'] = x_vals

        # ---- coarse z bins at the real resolution ----
        zedges = (np.arange(n_bins + 1) - n_bins / 2) * bin_nm             # e.g. -16, -8, 0, 8, 16
        centers = 0.5 * (zedges[:-1] + zedges[1:])                         # e.g. -12, -4, 4, 12
        nb = len(centers)
        width = 0.9 * bin_nm
        print(f"{nb} z bins of {bin_nm:g} nm, centers at {centers}")

        d['z_rel'] = d['z'] - z_center
        d['bin'] = np.digitize(d['z_rel'], zedges) - 1
        d = d[(d['bin'] >= 0) & (d['bin'] < nb)]
        grouped_b = d.groupby('bin')[angle_col].agg(['mean', 'std', 'count', 'sem']).reindex(range(nb))

 
        # ---- zero crossings per (x, y) column ----
        

        # z0 = z0_df['z0_rel_nm'].to_numpy()
        # counts, _ = np.histogram(z0, bins=zedges)
        # counts = pd.Series(counts, index=range(nb))

        y, _ = np.histogram(z0_df['z0_rel_nm'], bins=zedges)

        if normalize_bins: 
            y = y / y.sum() 


        # ---- statistics from the individual z0 values (independent of bin size) ----
        z0 = z0_df['z0_rel_nm'].to_numpy()
        z0 = z0[(z0 >= zedges[0]) & (z0 <= zedges[-1])]
        stats = {'n': len(z0)}
        if len(z0) > 1:
            stats.update(mean=float(np.mean(z0)), median=float(np.median(z0)),
                        variance=float(np.var(z0, ddof=1)), std=float(np.std(z0, ddof=1)), sterr = (float(np.std(z0, ddof=1)) / np.sqrt(len(z0))))
        else:
            stats = None


        # ---- Gaussian fit to the binned counts (always run, so params are saved) ----
        fit = None
        if show_fit and stats is not None and y.sum() > 0 and nb >= 4:
            try:
                p0 = [y.max(), stats['mean'], max(stats['std'], bin_nm)]
                (A, mu, sigma), _ = curve_fit(_gaussian, centers, y, p0=p0, maxfev=10000)
                sigma = abs(sigma)
                fwhm = 2 * np.sqrt(2 * np.log(2)) * sigma
                resid = y - _gaussian(centers, A, mu, sigma)
                ss_tot = np.sum((y - y.mean()) ** 2)
                r2 = 1 - np.sum(resid ** 2) / ss_tot if ss_tot > 0 else np.nan
                fit = dict(A=float(A), mu=float(mu), sigma=float(sigma),
                           fwhm=float(fwhm), r2=float(r2))
                stats.update(fit)
            except (RuntimeError, ValueError) as e:
                print(f"Gaussian fit failed for {name}: {e}")




        # ---- this sample's own layers and center ----
        # layers = np.arange(grouped_b.index.min(), grouped_b.index.max() + 1)
        # center = layers[layers.size // 2]
        # counts = z0_df['z0_rel_nm'].value_counts().reindex(range(nb), fill_value=0)
 
        p = {'comp': comp, 'color': color, 'grouped': grouped_b, 'counts': y,
             'x_line': centers,     # centered layer positions of the line plot
             'x_bar': centers, 
             'width': width, 
             'fit': fit}                          # same centering, full layer range for the bars
        plot_data.append(p)
        results[name] = {'grouped': grouped_b, 'zero_df': z0_df, 'stats': stats, 'fit': fit, 'counts': y}


        # print(len(counts))
        # print((centers))

        x_lo = p['x_bar'].min() 
        x_hi = p['x_bar'].max() 
        ticks1 = np.arange(x_lo, x_hi + bin_nm, bin_nm)


        if plot_individual:
            fig, (ax, ax2) = plt.subplots(1, 2, figsize=(22,7), sharex=True)
            draw(ax, ax2, p, line_alpha=1.0, bar_alpha=0.5)
            if show_fit:
                draw_fit(ax2, p, zedges)
            finish(ax, ax2, ticks1, f'{angle_col} vs. thickness position, {comp} \n {title}', plot_stats = stats)
 
    # ---- combined plot: all samples overlaid ----
    if plot_data:
        fig, (ax, ax2) = plt.subplots(2, 1, figsize=(25, 15), sharex=True)
        for p in plot_data:
            draw(ax, ax2, p, line_alpha=0.8, bar_alpha=0.4)

        fits = [p['fit'] for p in plot_data if p['fit'] is not None]

        if show_fit_summary and fits:
            mus = np.array([f['mu'] for f in fits])
            #ax2.axvline(np.mean(mus), color='orange', lw=1.5, label=f"mean μ = {np.mean(mus):.2f} nm")
            for name, c in zip(sample_names_plot, sample_colors):
                meds = results[name]['stats']['median']
                ax2.axvline(meds, color=c, lw=1.5, ls='-.',
                            label=f"median = {meds:.2f} nm")
                
            results['fit_summary'] = {
                'mu_mean': float(np.mean(mus)), 'mu_median': float(np.median(mus)),
                'sigma_mean': float(np.mean([f['sigma'] for f in fits])),
                'sigma_median': float(np.median([f['sigma'] for f in fits])),
                'fwhm_mean': float(np.mean([f['fwhm'] for f in fits])),
                'fwhm_median': float(np.median([f['fwhm'] for f in fits])),
            }
 
        x_lo = min(p['x_bar'].min() for p in plot_data)
        x_hi = max(p['x_bar'].max() for p in plot_data)
        finish(ax, ax2, np.arange(x_lo, x_hi + bin_nm, bin_nm),
               f'{angle_col} vs. thickness position, all samples \n {title}' )
 
    return results

In [ ]:
# mz_max = combined_df.loc[combined_df['sample'] == '270', 'Mz']
SN_Bloch_results = plot_Bloch_angle_vs_z(combined_df, combined_z0_df, sample_names, sample_names_plot, sample_colors, figure_name, bounds=(-16,16), angle_col='M_n_n', bloch_axis = [0] , neel_axis = [-1, 1], title = '', voxel_nm = voxel_nm, bin_nm = 8 )

# Curvature

In [ ]:
def _binned_raw_stats(bin_idx, z, w, cl, n_bins, k=None):
    """
    Non-bootstrap companion to _binned_cluster_stats, for checking its assumptions.

    bin_idx : int bin index per point; z : value; w : weight; cl : compact int cluster id
    k       : optional raw kappa per point; if given, also fits a weighted linear trend z ~ k

    Returns mean, err, n_pts, n_clusters, extra
      mean : weighted mean per bin (should match the bootstrap point estimate exactly)
      err  : naive SEM assuming independent points (std / sqrt(n_eff))
      extra (dict):
        std         : raw (weighted) standard deviation of z within each bin
        n_eff       : Kish effective sample size, (sum w)^2 / sum w^2
        sem_cluster : analytic cluster-robust SEM (ratio-estimator linearization),
                      the no-resampling analogue of the bootstrap err
        deff        : (sem_cluster / err)^2, design effect; ~1 means points act independent,
                      >>1 means within-cluster correlation is inflating the true error
        trend       : dict(slope, intercept, se_naive, se_cluster) if k is given
    """
    bin_idx = np.asarray(bin_idx)
    n_cl = cl.max() + 1

    # point-level per-bin sums
    Sw = np.bincount(bin_idx, weights=w, minlength=n_bins)
    Sw2 = np.bincount(bin_idx, weights=w**2, minlength=n_bins)
    Szw = np.bincount(bin_idx, weights=w * z, minlength=n_bins)
    n_pts = np.bincount(bin_idx, minlength=n_bins)

    # per-cluster, per-bin weight sums (for n_clusters and cluster-robust SE)
    Sw_cb = np.zeros((n_cl, n_bins))
    Szw_cb = np.zeros((n_cl, n_bins))
    np.add.at(Sw_cb, (cl, bin_idx), w)
    np.add.at(Szw_cb, (cl, bin_idx), w * z)
    n_clusters = (Sw_cb > 0).sum(0)

    with np.errstate(invalid='ignore', divide='ignore'):
        mean = Szw / Sw

        # two-pass weighted variance, with reliability-weight small-sample correction
        dev2 = (z - mean[bin_idx]) ** 2
        var_pop = np.bincount(bin_idx, weights=w * dev2, minlength=n_bins) / Sw
        n_eff = Sw**2 / Sw2
        std = np.sqrt(var_pop * n_eff / (n_eff - 1))
        err = std / np.sqrt(n_eff)

        # cluster-robust SEM: residual of each cluster's contribution to the ratio estimator
        resid = Szw_cb - mean * Sw_cb
        sem_cluster = np.sqrt(n_cl / (n_cl - 1) * (resid**2).sum(0)) / Sw
        deff = (sem_cluster / err) ** 2

    extra = dict(std=std, n_eff=n_eff, sem_cluster=sem_cluster, deff=deff)

    if k is not None:
        k = np.asarray(k, dtype=float)
        W = w.sum()
        kbar = (w * k).sum() / W
        zbar = (w * z).sum() / W
        kc = k - kbar
        Sxx = (w * kc**2).sum()
        slope = (w * kc * (z - zbar)).sum() / Sxx
        intercept = zbar - slope * kbar
        e = z - (intercept + slope * k)
        n = len(z)
        se_naive = np.sqrt((w * e**2).sum() / (n - 2) / Sxx)
        score_c = np.bincount(cl, weights=w * kc * e, minlength=n_cl)  # per-cluster score
        se_cluster = np.sqrt(n_cl / (n_cl - 1) * (score_c**2).sum()) / Sxx
        extra['trend'] = dict(slope=slope, intercept=intercept,
                              se_naive=se_naive, se_cluster=se_cluster)

    return mean, err, n_pts, n_clusters, extra


In [ ]:
def plot_z0_vs_curvature_all(combined_df, combined_z0_df, sample_names_plot, sample_colors,
                             n_bins_kappa=5, kappa_range=None, kappa_tol=0.002,
                             min_count=10, cluster_nm=20.0, normalize=True,
                             ylim=None, title=None, voxel_nm = 8.0, bin_nm=8.0):
    """
    Top left : mean z0 vs. kappa, one line per sample (shared kappa bins).
    Top right: mean z0 per curvature class, one marker per sample.
    Bottom   : z0 distribution for each class (kappa < -tol, |kappa| <= tol, kappa > tol),
               samples overlaid as step histograms (dashed = that sample's class mean).
    Returns (summary_all, class_summary_all, diff_df, data_by_sample).
    """
    names = [f'κ < -{kappa_tol:g}', f'|κ| ≤ {kappa_tol:g}', f'κ > {kappa_tol:g}']

    # ---- per-sample prep ----
    data = {}
    for s in sample_names_plot:
        d = combined_z0_df[combined_z0_df['sample'] == s].dropna(subset=['kappa']).copy()
        d['bx'] = np.floor(d['x'] / cluster_nm).astype(int)
        d['by'] = np.floor(d['y'] / cluster_nm).astype(int)
        d['cl'] = d.groupby(['bx', 'by']).ngroup()
        data[s] = d
        print(f"{s}: {len(d)} columns with valid kappa")

    pooled = pd.concat(data.values())

    # ---- shared kappa bins ----
    if kappa_range is None:
        lim = np.percentile(np.abs(pooled['kappa']), 98)
        kappa_range = (-lim, lim)
    print(f"kappa bins span {kappa_range[0]:.4f} to {kappa_range[1]:.4f} 1/nm")
    edges = np.linspace(*kappa_range, n_bins_kappa + 1)
    centers = 0.5 * (edges[:-1] + edges[1:])

    # ---- shared z bins, one bin centered on z = 0 ----
    ext = np.abs(pooled['z0_rel_nm']).max() + 1#+ bin_nm / 2
    print(ext)
    # n_half = int(np.ceil((ext - bin_nm / 2) / bin_nm))
    # zedges = (np.arange(-n_half, n_half + 2) - 0.5) * bin_nm
    # zcenters = 0.5 * (zedges[:-1] + zedges[1:])

    # z_center = 0.5 * (0 + ext)                    # film center (symmetric crop)
    T = ext * voxel_nm                      # film thickness covered by the samples
    n_bins = int(round(T / bin_nm))

    zedges = (np.arange(n_bins + 1) - n_bins / 2) * bin_nm             # e.g. -16, -8, 0, 8, 16
    zcenters = 0.5 * (zedges[:-1] + zedges[1:])   
    print(zcenters)


    # ---- figure ----
    fig = plt.figure(figsize=(16, 10))
    gs = fig.add_gridspec(2, 3)
    ax = fig.add_subplot(gs[0, :2])
    axc = fig.add_subplot(gs[0, 2])
    axh = [fig.add_subplot(gs[1, j]) for j in range(3)]
    for a in axh[1:]:
        a.sharex(axh[0]); a.sharey(axh[0])

    summaries, class_summaries = [], []
    n_s = len(sample_names_plot)
    offsets = (np.arange(n_s) - (n_s - 1) / 2) * 0.12

    for i, (s, color) in enumerate(zip(sample_names_plot, sample_colors)):
        d = data[s]
        z = d['z0_rel_nm'].to_numpy()
        k = d['kappa'].to_numpy()
        w = np.ones(len(d))
        cl = d['cl'].to_numpy()

        # kappa-binned means
        vb = pd.cut(pd.Series(k), edges, labels=False).to_numpy()
        vm = ~np.isnan(vb)
        mean, err, n_pts, n_cl, _ = _binned_raw_stats(
            vb[vm].astype(int), z[vm], w[vm], pd.factorize(cl[vm])[0], n_bins_kappa, k=k[vm])
        keep = n_pts >= min_count
        summaries.append(pd.DataFrame({'sample': s, 'kappa_center': centers, 'z_nm_mean': mean,
                                       'z_nm_err': err, 'n_points': n_pts,
                                       'n_clusters': n_cl, 'shown': keep}))
        ax.errorbar(centers[keep], (mean[keep]), yerr=err[keep], fmt='o-', capsize=3,
                    color=color, label=s, alpha=0.85)

        # class means
        cls = np.where(k < -kappa_tol, 0, np.where(k > kappa_tol, 2, 1))
        c_mean, c_err, c_n, c_ncl, _ = _binned_raw_stats(cls, z, w, cl, 3, k=None)
        class_summaries.append(pd.DataFrame({'sample': s, 'class': names, 'z_nm_mean': c_mean,
                                             'z_nm_err': c_err, 'n_points': c_n,
                                             'n_clusters': c_ncl}))
        axc.errorbar(np.arange(3) + offsets[i], (c_mean), yerr=c_err, fmt='o', capsize=3,
                     color=color, label=s)
  

        # per-class histograms
        for j in range(3):
            m = cls == j
            counts, _ = np.histogram(z[m], bins=zedges)
            if counts.sum() == 0:
                continue
            vals = counts / counts.sum() if normalize else counts
            axh[j].bar(zcenters, vals, width=bin_nm*0.9, color=color, alpha=0.4,
                       edgecolor=color, linewidth=1.2, label=f'{s} (n={m.sum()})')
            if np.isfinite(c_mean[j]):
                axh[j].axvline(c_mean[j], color=color, ls='--', lw=1.2)

    # ---- styling ----
    ax.axvline(0, color='k', ls=':', lw=0.8, label='kappa = 0')
    ax.axhline(0, color='gray', ls='--', lw=0.8, label='film center')
    ax.set_xlabel('dw curvature (1/nm)')
    ax.set_ylabel('Bloch position rel. to film center (nm)')
    ax.set_title(f'Mean Bloch position (z) vs. curvature\n{title}')
    ax.legend(fontsize=8)
    if ylim is not None:
        ax.set_ylim(*ylim)

    axc.axhline(0, color='gray', ls='--', lw=0.8)
    axc.set_xticks(range(3))
    axc.set_xticklabels(names)
    axc.set_ylabel('mean Bloch position (nm)')
    axc.set_title('Class means by sample')
    axc.legend(fontsize=8)

    for j, a in enumerate(axh):
        a.axvline(0, color='k', ls=':', lw=0.8)
        a.set_title(names[j])
        a.set_xlabel('z relative to film center (nm)')
        a.set_xticks(zcenters)
        a.set_xticklabels([f'{v:g}' for v in zcenters], rotation=45)
        a.legend(fontsize=8)
    axh[0].set_ylabel('fraction of columns' if normalize else 'counts')

    plt.tight_layout()
    plt.show()

    # ---- tables ----
    summary_all = pd.concat(summaries, ignore_index=True)
    class_summary_all = pd.concat(class_summaries, ignore_index=True)

    pv_m = class_summary_all.pivot(index='sample', columns='class', values='z_nm_mean')[names]
    pv_e = class_summary_all.pivot(index='sample', columns='class', values='z_nm_err')[names]
    diff_df = pd.DataFrame({
        'delta_z_nm': pv_m[names[2]] - pv_m[names[0]],
        'err': np.sqrt(pv_e[names[2]] ** 2 + pv_e[names[0]] ** 2),
    }).loc[sample_names_plot]
    diff_df.columns = [f'mean z0({names[2]}) - mean z0({names[0]})', 'err (quadrature)']

    print("\nClass summary:")
    print(class_summary_all.to_string(index=False))
    print("\nPositive-vs-negative curvature shift per sample:")
    print(diff_df.to_string())
    return summary_all, class_summary_all, diff_df, data

In [ ]:
summary_all, class_all, diff_df, data = plot_z0_vs_curvature_all( combined_df,
    combined_z0_df, sample_names_plot, sample_colors, n_bins_kappa=5,
    kappa_tol=0.005, normalize=True, title='', voxel_nm = voxel_nm, bin_nm=voxel_nm)

In [ ]:
summary_all, class_all, diff_df, data = plot_z0_vs_curvature_all( combined_df,
    combined_z0_df, sample_names_plot, sample_colors, n_bins_kappa=7,
    kappa_tol=0.005, normalize=True, title='', voxel_nm = voxel_nm, bin_nm=voxel_nm)

In [ ]:
def plot_z0_classes_per_sample(data, summary_all, sample_names_plot, sample_colors, kappa_tol=0.002, bin_nm=8.0, voxel_nm = 8.0, 
                               normalize=True, grouped=True, title=None):
    """
    One figure per sample; the three kappa classes are compared in the same axes.
    data : dict {sample: DataFrame} returned by plot_z0_vs_curvature_all.
    grouped=True draws the classes side by side in each bin instead of overlaid.
    """
    names = [f'κ < -{kappa_tol:g}', f'|κ| ≤ {kappa_tol:g}', f'κ > {kappa_tol:g}']
    colors = ['tab:blue', 'tab:gray', 'tab:red']

    # shared z bins across samples, one bin centered on z = 0
    pooled = pd.concat(data.values())
    # ext = np.abs(pooled['z0_rel_nm']).max() + bin_nm / 2
    # n_half = int(np.ceil((ext - bin_nm / 2) / bin_nm))
    # zedges = (np.arange(-n_half, n_half + 2) - 0.5) * bin_nm
    # zcenters = 0.5 * (zedges[:-1] + zedges[1:])

    ext = np.abs(pooled['z0_rel_nm']).max() + 1#+ bin_nm / 2
    T = ext * voxel_nm                      # film thickness covered by the samples
    n_bins = int(round(T / bin_nm))
    zedges = (np.arange(n_bins + 1) - n_bins / 2) * bin_nm             # e.g. -16, -8, 0, 8, 16
    zcenters = 0.5 * (zedges[:-1] + zedges[1:])   
    # print(zcenters)


    # for s in sample_names_plot:
    for i, (s, color) in enumerate(zip(sample_names_plot, sample_colors)):
        print(s)
        d = data[s]
        line_plot =  summary_all[summary_all['sample'] == s]
        z = d['z0_rel_nm'].to_numpy()
        k = d['kappa'].to_numpy()
        w = np.ones(len(d))
        cl = d['cl'].to_numpy()
        cls = np.where(k < -kappa_tol, 0, np.where(k > kappa_tol, 2, 1))
        c_mean, c_err, c_n, c_ncl, _ = _binned_raw_stats(cls, z, w, cl, 3, k=None)

        fig, (ax, axh) = plt.subplots(1, 2, figsize=(13, 4.5))
        ax.errorbar(line_plot['kappa_center'], line_plot['z_nm_mean'], yerr=line_plot['z_nm_err'], fmt='o-', capsize=3, color=color)
        ax.axvline(0, color='k', ls=':', lw=0.8, label='kappa = 0')
        ax.axhline(0, color='gray', ls='--', lw=0.8, label='film center')
        
        for j, c in enumerate(colors):
            ax.axhline(c_mean[j], color=c, ls='--', lw=1.5)



        # ax.axvspan(-kappa_tol, kappa_tol, color='gray', alpha=0.12, label=f'|κ| ≤ {kappa_tol:g}')
        ax.set_xlabel('dw curvature (1/nm)')
        ax.set_ylabel('Bloch position relative to film center (nm)')
        ax.set_title(f'Mean Bloch position (z) vs. curvature \n {s}')
        ax.legend(fontsize=8)



        for j, c in enumerate(colors):
            m = cls == j
            counts, _ = np.histogram(z[m], bins=zedges)
            if counts.sum() == 0:
                continue
            vals = counts / counts.sum() if normalize else counts

            if grouped:
                wb = bin_nm * 0.9 / 3
                x = zcenters + (j - 1) * wb
                width = wb
            else:
                x = zcenters
                width = bin_nm * 0.9

            axh.bar(x, vals, width=width, color=c, edgecolor=c,
                   linewidth=1.2, label=f'{names[j]} (mean ={c_mean[j]:.2f})')
            if np.isfinite(c_mean[j]):
                axh.axvline(c_mean[j], color=c, ls='--', lw=1.5)

        axh.axvline(0, color='k', ls=':', lw=0.8)
        axh.set_xlabel('z relative to film center (nm)')
        axh.set_ylabel('fraction of columns' if normalize else 'counts')
        axh.set_xticks(zcenters)
        axh.set_xticklabels([f'{v:g}' for v in zcenters], rotation=45)
        axh.set_title(f'Bloch position distribution by curvature, {s}\n{title or ""}')
        axh.legend(fontsize=9)
        plt.tight_layout()
        plt.show()

In [ ]:
plot_z0_classes_per_sample(data, summary_all, sample_names_plot, sample_colors, kappa_tol=0.005, bin_nm=voxel_nm, voxel_nm = voxel_nm, normalize=True)

# DW width 

In [ ]:
def contour_curvature(field, level=0.0, smoothing=2.0, n_eval=400, min_points=8,
                       pixel_size=1.0):
    """
    Extract the domain boundary/boundaries of `field` at `level` and
    compute signed curvature along each.

    Parameters
    ----------
    field : (H, W) array
        Scalar field, e.g. your Mz array. Sign of (field - level) defines
        the two domains.
    level : float, default 0.0
        Threshold defining the boundary. 0.0 is the natural choice for a
        field like Mz that is symmetric about zero.
    smoothing : float, default 2.0
        Spline smoothing strength, passed to scipy as `s = smoothing *
        n_points_in_contour`. Larger values fit a smoother curve (less
        sensitive to pixel staircasing, but can round off genuinely sharp
        features); smaller values track the raw pixel boundary more
        closely (noisier curvature). Start around 1-3 and adjust by eye
        against the plot. Applied in pixel-index space, so this doesn't
        need to change when you change `pixel_size`.
    n_eval : int, default 400
        Number of points each contour is resampled to for evaluating
        curvature (independent of the original pixel contour length).
    min_points : int, default 8
        Contours with fewer raw points than this are skipped (too short
        to reliably spline-fit; usually stray single-pixel noise).
    pixel_size : float, default 1.0
        Physical length per pixel (e.g. 2.0 for 2 nm/pixel). Curvature
        scales as 1/pixel_size and coordinates scale as pixel_size, so
        with the default of 1.0 everything stays in pixel/voxel units
        exactly as before. Assumes isotropic (square) pixels.

    Returns
    -------
    list of dicts, one per contour, each with:
        'xy'    : (n_eval, 2) array of (x, y) coordinates in physical
                  units (x = column * pixel_size, y = row * pixel_size),
                  suitable for plotting over `field` with imshow(...,
                  origin='lower', extent=[0, W*pixel_size, 0, H*pixel_size]).
        'kappa' : (n_eval,) array of signed curvature, in 1/[pixel_size units].
    """
    ## find contour between up and down domains 
    contours = find_contours(field, level=level)
    results = []
    half_width = 10      # in pixels, how far to sample on each side of the wall
    n_samples  = 41      # points along each normal
    s = np.linspace(-half_width, half_width, n_samples) 


    
    for c in contours:
        if len(c) < min_points:
            continue
        rows, cols = c[:, 0], c[:, 1]
        closed = np.hypot(rows[0] - rows[-1], cols[0] - cols[-1]) < 1.0
        per = 1 if closed else 0
        try:
            ## define a spline of the curve (for each contour, input x y values and fit with a cubic (3) spline)
            ##  a spline is a function defined in pieces by polynomials that stitch together neighboring points smoothly
            tck, _ = splprep([cols, rows], s=smoothing * len(c), per=per, k=3)
        except Exception:
            continue

        uu = np.linspace(0, 1, n_eval) ## segments points on the contour from 0 being the beginning of the line and 1 being the end of the line 
        x, y = splev(uu, tck, der=0) ## evaluated spline and get x y positions
        dx, dy = splev(uu, tck, der=1) ## first derivative of the spline -----> This is the unit tangent. Rotate this 90 deg. to get the normal 
        ddx, ddy = splev(uu, tck, der=2) ## second derivative of the spline


        # unit tangent and unit normal (x = column, y = row)
        speed = np.hypot(dx, dy)
        speed = np.where(speed == 0, 1e-12, speed)
        tx, ty = dx / speed, dy / speed
        nx, ny = -ty, tx                     # tangent rotated +90 degrees == the surface normal 

        ## plane curve definition of curvature (https://en.wikipedia.org/wiki/Curvature)
        denom = (dx**2 + dy**2) ** 1.5
        kappa = (dx * ddy - dy * ddx) / np.where(denom == 0, 1e-12, denom)
        # convert pixel-index-space quantities to physical units
        kappa = kappa / pixel_size
        xy = np.stack([x, y], axis=1) * pixel_size
        results.append({"xy": xy, "kappa": kappa, "normal": np.stack([nx, ny], axis=1)})
    return results

In [ ]:
def _sech(x):
    return 1 / np.cosh(x)

MODELS = {
    # name: (func, extra_p0, extra_lower, extra_upper)
    "gauss":   (lambda s, A, mu, w, off: A*np.exp(-0.5*((s-mu)/w)**2) + off, [], [], []),
    "sech2":   (lambda s, A, mu, w, off: A*_sech((s-mu)/w)**2 + off,         [], [], []),
    "sech":    (lambda s, A, mu, w, off: A*_sech((s-mu)/w) + off,            [], [], []),

}


# FWHM = factor * w   (for the Gaussian, w is sigma)
FWHM_FACTOR = {"gauss": 2*np.sqrt(2*np.log(2)),      # 2.355
               "sech":  2*np.arccosh(2),             # 2.634
               "sech2": 2*np.arccosh(np.sqrt(2))}    # 1.763
LABEL = {"gauss": "Gaussian", "sech": "sech", "sech2": "sech²"}


def fit_peak(s, prof, model="gauss"):
    """Fit A*shape((s-mu)/w) + offset. Returns the same keys for every model."""
    func, k = MODELS[model][0], FWHM_FACTOR[model]
    nan = np.nan
    out = dict(success=False, A=nan, mu=nan, w=nan, offset=nan, fwhm=nan, rms=nan,
               mu_err=nan, w_err=nan, fwhm_err=nan)
    ok = np.isfinite(prof)
    if ok.sum() < 6:
        return out
    s_, p_ = s[ok], prof[ok]
    ds = s_[1] - s_[0]
    j = np.argmax(p_)
    off0 = p_.min()
    A0 = p_[j] - off0
    above = p_ > off0 + A0 / 2
    width0 = s_[above].max() - s_[above].min() if above.sum() > 1 else 4 * ds
    w0 = max(width0 / k, ds)                      # start from the half-max width

    try:
        popt, pcov = curve_fit(func, s_, p_, p0=[A0, s_[j], w0, off0],
                               bounds=([0, s_[0], ds/2, -np.inf],
                                       [np.inf, s_[-1], s_[-1] - s_[0], np.inf]),
                               maxfev=20000)
    except (RuntimeError, ValueError):
        return out
    perr = np.sqrt(np.diag(pcov))
    out.update(success=True, A=popt[0], mu=popt[1], w=popt[2], offset=popt[3],
               fwhm=k * popt[2], mu_err=perr[1], w_err=perr[2], fwhm_err=k * perr[2],
               rms=np.sqrt(np.mean((p_ - func(s_, *popt)) ** 2)), f=func, popt=popt)
    return out

In [ ]:
def compute_profiles(gradient, xy, normal, pixel_size, half_width, n_samples=81, order=3):
    """
    Sample `img` along the normal at each point of a contour.

    img        : 2D array (e.g. gradient magnitude), indexed [row, col]
    xy         : (N, 2) physical positions (x=col, y=row) * pixel_size
    normal     : (N, 2) unit normals (nx, ny)
    half_width : physical distance to sample on each side of the contour
    Returns
    s          : (n_samples,) signed physical distance along the normal
    profiles   : (N, n_samples) sampled values
    XY         : (N, n_samples, 2) physical sample positions

    """
    s = np.linspace(-half_width, half_width, n_samples)

    Xp = xy[:, 0][:, None] + s[None, :] * normal[:, 0][:, None]   # physical
    Yp = xy[:, 1][:, None] + s[None, :] * normal[:, 1][:, None]

    # map_coordinates wants pixel-index (row, col) = (Y, X)
    profiles = map_coordinates(gradient, [Yp / pixel_size, Xp / pixel_size],
                               order=order, mode="nearest")
    return s, profiles, np.stack([Xp, Yp], axis=-1)

def fwhm_from_curve(s, y, baseline):
    """Half-max width by linear interpolation of the crossings around the peak."""
    j = np.nanargmax(y)
    half = baseline + (y[j] - baseline) / 2
    l = j
    while l > 0 and y[l] > half:
        l -= 1
    r = j
    while r < len(y) - 1 and y[r] > half:
        r += 1
    if y[l] > half or y[r] > half:        # never dropped below half max
        return np.nan, np.nan, np.nan
    sl = s[l] + (half - y[l]) * (s[l+1] - s[l]) / (y[l+1] - y[l])
    sr = s[r-1] + (half - y[r-1]) * (s[r] - s[r-1]) / (y[r] - y[r-1])
    return sr - sl, sl, sr

def fwhm_along_contour(gradient, result, pixel_size, half_width,
                       n_samples=81, order=3, model="gauss"):
    """Fit `model` to the normal profile at every point of one contour."""
    s, profiles, XY = compute_profiles(gradient, result["xy"], result["normal"],
                                       pixel_size, half_width, n_samples, order)
    N = len(profiles)
    ny, nx = gradient.shape
    inside = ((XY[..., 0] >= 0) & (XY[..., 0] <= (nx - 1) * pixel_size) &
              (XY[..., 1] >= 0) & (XY[..., 1] <= (ny - 1) * pixel_size)).all(axis=1)

    keys = ["fwhm", "w", "mu", "A", "offset", "rms", "fwhm_err", "mu_err", "w_err"]
    out = {k: np.full(N, np.nan) for k in keys}
    out["fwhm_direct"] = np.full(N, np.nan)
    out["success"] = np.zeros(N, bool)

    for j, p in enumerate(profiles):
        if not (inside[j] and np.all(np.isfinite(p))):
            continue
        out["fwhm_direct"][j] = fwhm_from_curve(s, p, p.min())[0]   # model-free
        f = fit_peak(s, p, model)
        if f["success"]:
            out["success"][j] = True
            for k in keys:
                out[k][j] = f[k]

    out.update(inside=inside, pt=np.arange(N), half_width=half_width, model=model)
    return out

def fwhm_slice(grad2d, contours, pixel_size, half_width, min_good=20, model="gauss"):
    vals, cmeans, used = [], [], []
    n_pts = 0
    for ci, r in enumerate(contours):
        fw = fwhm_along_contour(grad2d, r, pixel_size, half_width, model=model)
        good = fw["success"] & fw["inside"] & (fw["fwhm"] < fw["half_width"])
        n_pts += len(good)
        if good.sum() < min_good:
            continue
        v = fw["fwhm"][good]
        vals.append(v); cmeans.append(v.mean())
        used.append(dict(idx=ci, xy=r["xy"], mean=v.mean(), n_good=int(good.sum())))
    v = np.concatenate(vals) if vals else np.array([])
    sd = lambda a: np.std(a, ddof=1) if len(a) > 1 else np.nan
    return dict(n_good=len(v), n_pts=n_pts, n_contours=len(used),
                mean=v.mean() if len(v) else np.nan,
                median=np.median(v) if len(v) else np.nan,
                std=sd(v), stderr=sd(v) / np.sqrt(len(v)) if len(v) > 1 else np.nan,
                vals=v, contour_means=np.array(cmeans), contours=used)


def fwhm_vs_z_slices(Mz3, grad3, contour_fn, pixel_size, half_width,
                     z_indices=None, min_good=20, model="gauss", verbose=False):
    z_indices = range(len(Mz3)) if z_indices is None else z_indices
    rows = {}
    for k in z_indices:
        rows[k] = fwhm_slice(grad3[k], contour_fn(Mz3[k]), pixel_size, half_width,
                             min_good, model=model)
        if verbose:
            r = rows[k]
            print(f"  z={k}: {r['n_contours']} contours, {r['n_good']}/{r['n_pts']} points")
    return rows

In [ ]:
stop

In [ ]:
from scipy.interpolate import splprep, splev

voxel_nm = 8.0                      # in-plane pixel size (nm); or a dict {name: value}
dz_nm = 8.0                         # slice spacing (nm);       or a dict {name: value}
half = 5.0                # profile half-width in nm (see note below)
model = "gauss"

get = lambda v, n: v[n] if isinstance(v, dict) else v

all_rows = {}
for name, label in zip(sample_names, sample_names_plot):
    Mz_slice = volumes[name]['Mz'][2]
    px = get(voxel_nm, name)
    cfn = lambda Mz_slice, px=px: contour_curvature(Mz_slice, level=0.0, smoothing=0.1,
                                              n_eval=400, pixel_size=px)
    print(label)
    all_rows[label] = fwhm_vs_z_slices(volumes[name]['Mz'], grad[name]['grad_xy'], cfn,
                                       voxel_nm, half * voxel_nm, model=model, verbose=True)

In [ ]:
def bin_slice_fwhm_by_z(rows, dz_nm, bin_nm=8.0):
    zs = np.array(sorted(rows))
    bins = zs if bin_nm is None else np.floor(zs * dz_nm / bin_nm + 1e-9).astype(int)
    sd = lambda a: np.std(a, ddof=1) if len(a) > 1 else np.nan
    out = []
    for b in np.unique(bins):
        ks = zs[bins == b]
        v = np.concatenate([rows[k]["vals"] for k in ks])
        v = v[np.isfinite(v)]
        cm = np.concatenate([rows[k]["contour_means"] for k in ks])
        if len(v) == 0:
            continue
        out.append(dict(
            center=ks[0] * dz_nm if bin_nm is None else (b + 0.5) * bin_nm,
            slices=ks, n=len(v), n_contours=len(cm),
            mean=v.mean(), median=np.median(v), std=sd(v),
            stderr=sd(v) / np.sqrt(len(v)) if len(v) > 1 else np.nan,
            stderr_contour=sd(cm) / np.sqrt(len(cm)) if len(cm) > 1 else np.nan))
    return out

def plot_fwhm_vs_z_all(all_rows, dz_nm, colors, bin_nm=8.0, err="stderr", stat="mean",
                       dodge=0.08, cmap="tab10", unit="nm", model="gauss", min_n=1):
    cm = plt.get_cmap(cmap)
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = list(all_rows)
    binned_all = {}
    for j, label in enumerate(labels):
        b = bin_slice_fwhm_by_z(all_rows[label], get(dz_nm, label), bin_nm)
        b = [d for d in b if d["n"] >= min_n]
        binned_all[label] = b
        if not b:
            continue
        x = np.array([d["center"] for d in b]) + (j - (len(labels) - 1) / 2) * dodge * bin_nm
        y = np.array([d[stat] for d in b])
        e = np.array([d[err] for d in b])
        ax.errorbar(x, y, yerr=e, fmt="o-", ms=5, lw=1, capsize=3, color=colors[j], label=label)
    ax.set_xlabel(f"z ({unit}), {bin_nm:g} {unit} bins")
    ax.set_ylabel(f"FWHM of {model} fit ({unit})")
    ax.set_title(f"Domain wall FWHM vs. z, all contours ({stat} ± {err})")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig, ax, binned_all




fig, ax, binned_all = plot_fwhm_vs_z_all(all_rows, dz_nm, sample_colors, bin_nm=8.0, err="stderr", model=model)

In [ ]:
def _to_arrays(b):
    """list of per-bin dicts -> dict of arrays (drops the 'slices' array column)."""
    return {k: np.array([d[k] for d in b]) for k in b[0] if k != 'slices'}

def bin_rows_centered(rows, dz_nm, bin_nm=8.0, z_center_idx=None):
    """
    Pool slices into z bins centered on multiples of bin_nm, with z = 0 at the film center.
    z_center_idx : slice index (can be fractional) of the film center.
                   None -> middle of the analyzed slices.
    """
    ks = np.array(sorted(rows))
    if z_center_idx is None:
        z_center_idx = 0.5 * (ks.min() + ks.max())
    z = (ks - z_center_idx) * dz_nm
    bins = np.floor(z / bin_nm + 0.5 + 1e-9).astype(int)
    sd = lambda a: np.std(a, ddof=1) if len(a) > 1 else np.nan
    out = []
    for b in np.unique(bins):
        sel = ks[bins == b]
        v = np.concatenate([rows[k]["vals"] for k in sel])
        v = v[np.isfinite(v)]
        cm = np.concatenate([rows[k]["contour_means"] for k in sel])
        if len(v) == 0:
            continue
        out.append(dict(
            center=b * bin_nm, slices=sel, n=len(v), n_contours=len(cm),
            mean=v.mean(), median=np.median(v), std=sd(v),
            stderr=sd(v) / np.sqrt(len(v)) if len(v) > 1 else np.nan,
            stderr_contour=sd(cm) / np.sqrt(len(cm)) if len(cm) > 1 else np.nan))
    return out


def plot_fwhm_vs_z_all(all_rows, dz_nm, colors, bin_nm=8.0, err="stderr", stat="mean",
                       dodge=0.08, unit="nm", model="gauss", min_n=1,
                       z_center_idx=None):
    """
    z_center_idx : None, a scalar, or a dict {label: slice index of the film center}
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = list(all_rows)
    binned_all = {}
    binned_arr = {} 
    widths     = {}  

    for j, label in enumerate(labels):
        zc = z_center_idx[label] if isinstance(z_center_idx, dict) else z_center_idx
        b = bin_rows_centered(all_rows[label], get(dz_nm, label), bin_nm, zc)
        b = [d for d in b if d["n"] >= min_n]
        binned_all[label] = b
        if not b:
            continue
        binned_arr[label] = _to_arrays(b)

        z = binned_arr[label]['center']
        y = binned_arr[label][stat]
        e = binned_arr[label][err]

        i_bloch = np.argmin(np.abs(z))

        # Neel: first two + last two bins (the surface caps)
        n_edge = min(2, len(y))
        i_neel = np.unique(np.r_[0:n_edge, len(y) - n_edge:len(y)])   # e.g. [0, 1, 7, 8]

        bloch     = y[i_bloch]
        bloch_err = e[i_bloch]
        neel      = y[i_neel].mean()
        neel_err  = np.sqrt(np.sum(e[i_neel]**2)) / len(i_neel)       # error of the mean

        widths[label] = dict(
            bloch=bloch, bloch_err=bloch_err, z_bloch=z[i_bloch],
            neel=neel,   neel_err=neel_err,   n_neel_bins=len(i_neel),
            delta=neel - bloch,
            delta_err=np.hypot(neel_err, bloch_err),
        )

        x = z + (j - (len(labels) - 1) / 2) * dodge * bin_nm
        ax.errorbar(x, y, yerr=e, fmt="o-", ms=5, lw=1, capsize=3,
                    color=colors[j], label=label)

    ax.axvline(0, color="gray", ls=":", lw=1)
    ax.set_xlabel(f"z relative to film center ({unit}), {bin_nm:g} {unit} bins")
    ax.set_ylabel(f"FWHM of {model} fit ({unit})")
    ax.set_title(f"Domain wall FWHM vs. z, all contours ({stat} ± {err})")
    ax.legend(fontsize=8)
    fig.tight_layout()
    return fig, ax, binned_arr, binned_all, widths

fig, ax, binned_arr, binned_all, widths = plot_fwhm_vs_z_all(all_rows, dz_nm, sample_colors, bin_nm=8.0, err="stderr", model=model)

In [ ]:
widths['[Co(0.2 nm)/ Ni(0.7 nm)]'].keys()

# probability of Neel vs Bloch 

# Extracted values vs thickness etc. 


In [ ]:
binned_all['[Co(0.2 nm)/ Ni(0.7 nm)]'].keys()

In [ ]:
sw_res['270']['scalars'].keys()

In [ ]:
def _get(res, *keys):
    for k in keys:
        if k in res:
            return res[k]
    raise KeyError(f"none of {keys} found in {list(res)}")

params = {}
for i, name in enumerate(sample_names):
    name_2 = sample_names_plot[i]
    print(name_2)
    p = dict(keff[name])
    p['thickness_nm'] = sample_thickness[i]
    p['delta']        = widths[name_2]['delta']
    p['color']        = sample_colors[i]
    p['SN_bloch_mean'] = SN_Bloch_results[name_2]['stats']['mean']
    p['SN_bloch_median'] = SN_Bloch_results[name_2]['stats']['median']
    p['bloch']        = widths[name_2]['bloch']
    p['neel']        = widths[name_2]['neel']
    p['Aex']          = (((widths[name_2]['bloch']*1e-9) / np.pi)**2)*p['Keff'] * 1e12

    p.update(_get(sw_res, name, name_2)['scalars'])   # hel_switch_count, hel_switch_per_um2, ...
    p.update(_get(sw2x_res, name, name_2)['scalars'])   # hel_switch_count, hel_switch_per_um2, ...
    # curl stats: found under either naming convention
    p.update(_get(par_res, name, name_2)['scalars'])   # parallel_mean, _std, _sterr, _z_mean, _z_median
    p.update(_get(mag_res, name, name_2)['scalars'])   # curl_magnitude_mean, ...
    p.update(_get(hel_res, name, name_2)['scalars']) 
    p.update(_get(neel_res, name, name_2)['scalars'])

    params[name]      = p


# ── 2. Axis labels and reference lines (edit here to add new keys) ──
LABELS = {
    'thickness_nm': 'Sample thickness (nm)',
    'Ms':    r'$M_s$  (emu cm$^{-3}$)',
    'Hc':    r'$H_c$  (Oe)',
    'Mr':    r'$M_r$  (emu cm$^{-3}$)',
    'MrMs':  r'$M_r / M_s$',
    'Keff':  r'$K_\mathrm{eff}$  (J m$^{-3}$)',
    'Q':     r'$Q$',
    'delta': r'$\Delta_{DW}$  (nm)',
    'bloch': r'Bloch Width  (nm)',
    'neel': r'Neel Width  (nm)',
    'SN_bloch_mean': r'Mean Bloch position (SN) (nm)',
    'SN_bloch_median': r'Mean Bloch position (SN) (nm)',
    'Aex': r'$A_{ex}$ (pJ/m)',
    'parallel_z_mean': r'Mean Bloch position (curl parallel) (nm)',
    'curl_magnitude_z_mean': r'Mean Bloch position (curl mag) (nm)',
    'helicity_mean_at_center': r'dominating Bloch helicity',
    'hel_switch_red_x2_per_um2': f'Bloch helicity switch per μm$^2$', 
    'hel_switch_per_um2': f'Bloch helicity switch per μm$^2$', 

    'neel_bottom_n_red_per_um2': f'# minority helicity bottom per  μm$^2$', 
    'neel_top_n_blue_per_um2': f'# minority helicity top per μm$^2$', 


}
REF_LINES = {'Q': 1, 'Keff': 0, 'SN_bloch_mean': 0, 'SN_bloch_median': 0,'Aex': 0, 'parallel_z_mean': 0, 'curl_magnitude_z_mean': 0, 'helicity_mean_at_center' : 0}      # dotted guide line where each quantity changes meaning


In [ ]:
params.keys()

In [ ]:
print(params['270']['Aex'])
print(params['105']['Aex'])
print(params['665']['Aex'])
print(params['452']['Aex'])

In [ ]:
def param_plot(y, x='thickness_nm', ylabel=None, xlabel=None, title=None,
               names='top', connect=False, figsize=(7, 5)):
    """
    y, x   : keys in params (e.g. 'Keff', 'Q', 'delta', 'thickness_nm')
    names  : 'top' (second x-axis with sample names), 'annotate' (text by each point), or None
    connect: draw a gray line through the points (sorted by x)
    """
    order = sorted(params, key=lambda s: params[s]['thickness_nm'])
    xv  = np.array([params[s].get(x, np.nan) for s in order], float)
    yv  = np.array([params[s].get(y, np.nan) for s in order], float)
    col = [params[s]['color'] for s in order]
    ok  = ~(np.isnan(xv) | np.isnan(yv))          # skip samples missing either value

    fig, ax = plt.subplots(figsize=figsize)
    if connect:
        srt = np.argsort(xv[ok])
        ax.plot(xv[ok][srt], yv[ok][srt], color='gray', lw=1, zorder=1)
    ax.scatter(xv[ok], yv[ok], c=np.array(col, dtype=object)[ok].tolist(),
               s=100, edgecolors='k', linewidths=0.6, zorder=5)

    if y in REF_LINES: ax.axhline(REF_LINES[y], color='k', lw=0.6, ls=':')
    if x in REF_LINES: ax.axvline(REF_LINES[x], color='k', lw=0.6, ls=':')

    ax.set_xlabel(xlabel or LABELS.get(x, x), fontsize=16)
    ax.set_ylabel(ylabel or LABELS.get(y, y), fontsize=16)
    ax.tick_params(labelsize=13)
    if title: ax.set_title(title, fontsize=14)

    names_ok = np.array(order)[ok]
    if names == 'top':
        ax2 = ax.twiny()
        ax2.set_xlim(ax.get_xlim())
        ax2.set_xticks(xv[ok])
        ax2.set_xticklabels(names_ok, fontsize=12)
    elif names == 'annotate':
        for xi, yi, n in zip(xv[ok], yv[ok], names_ok):
            ax.annotate(n, (xi, yi), textcoords='offset points', xytext=(6, 6), fontsize=11)

    plt.tight_layout()
    plt.show()

# ── 4. Examples ──────────────────────────────────────────────
# vs thickness (default)
# param_plot('Ms')
# param_plot('Hc')
# param_plot('Mr')
# param_plot('Keff')
# param_plot('Q')
# param_plot('Q', names='annotate')
# param_plot('bloch', names='annotate')
# param_plot('neel', names='annotate')
# param_plot('SN_bloch_mean', names = 'annotate')

# # vs anything else
# param_plot('delta', x='Q',    names='annotate')
# param_plot('delta', x='Ms', names='annotate')
# param_plot('SN_bloch_mean', x = 'delta', names = 'annotate')

# param_plot('delta', x='Q',    names='annotate')
# param_plot('SN_bloch_mean', x = 'Q', names = 'annotate')
# param_plot('bloch', x = 'Q',  names='annotate')
# param_plot('neel', x = 'SN_bloch_mean', names = 'annotate')

# param_plot('Aex',  x='thickness_nm', names='annotate')
# param_plot('Aex',  x='Q', names='annotate')
# param_plot('Aex',  x='delta', names='annotate')
# param_plot('Aex',  x='bloch', names='annotate')

param_plot('parallel_z_mean',  x='thickness_nm', names='annotate')
param_plot('parallel_z_mean',  x='Q', names='annotate')
# param_plot('curl_magnitude_z_mean',  x='Q', names='annotate')
# param_plot('Aex',  x='parallel_z_mean', names='annotate')

# param_plot('parallel_z_mean',  x='Q', names='annotate')
# param_plot('curl_magnitude_z_mean',  x='Q', names='annotate')
# param_plot('Aex',  x='parallel_z_mean', names='annotate')


# param_plot('helicity_mean_at_center',  names='annotate')
# param_plot('hel_switch_red_x2_per_um2',  x='Q', names='annotate')
# param_plot('hel_switch_per_um2',  x='Q', names='annotate')

# param_plot('neel_bottom_n_red_per_um2',  x='Q', names='annotate')
# param_plot('neel_top_n_blue_per_um2',  x='Q', names='annotate')

# param_plot('neel_bottom_n_red_per_um2',   names='annotate')
# param_plot('neel_top_n_blue_per_um2',   names='annotate')

# param_plot('D_thr',  x='thickness_nm', names='annotate')
# param_plot('D_thr',  x='Q', names='annotate')
# param_plot('D_thr',  x='delta', names='annotate')
# param_plot('D_thr',  x='bloch', names='annotate')
